# Aula 03 · RCA assistido por IA

## 1. Da triagem à investigação

**Já sabemos que houve impacto. Agora queremos investigar o mecanismo.** Vamos usar a triagem do Pix, consultar documentação e construir uma análise que uma pessoa consiga revisar.

| De onde partimos | O que acrescentamos | O que entregamos |
|---|---|---|
| 80 falhas em 1.000 solicitações: **8%**. Duplicata reconhecida e sinais relacionados. | Trace, documentação recuperada e confronto de hipóteses com IA. | Linha do tempo, matriz de hipóteses e postmortem para revisão. |

**RCA** significa *Root Cause Analysis*, análise de causa raiz. O nome da prática não confirma uma causa: a conclusão precisa respeitar as evidências e as lacunas.

### Como percorrer este notebook

| Bloco | Pergunta que vamos responder | Seções |
|---|---|---|
| **Preparar** | Onde estão os arquivos e qual provedor vou usar? | 2 |
| **Entender o caso** | O que sabemos, quando soubemos e o que o trace mostra? | 3–6 |
| **Recuperar documentação** | Como um runbook vira contexto para o modelo? | 7–8 |
| **Analisar e revisar** | Onde a IA é chamada e como conferimos sua resposta? | 9–11 |
| **Construir** | Como acrescentar verificações ao grafo e aprofundar o RAG? | 12–14 |

<img src="projeto/assets/01-progressao.png" width="760" style="max-width:100%;height:auto" alt="Resumo, triagem, investigação e revisão.">

[Fonte editável](projeto/assets/01-progressao.mmd).

| O que já aprendemos | Como será usado aqui |
|---|---|
| Aula 01: organizar e resumir evidências. | Ler fontes e explicitar lacunas. |
| Aula 02: transformar triagem em regras. | Receber o dossiê e o contexto preparado, preservando os cálculos. |
| Ollama: chamar um modelo local dentro de um grafo. | Escolher OpenAI ou Ollama para o mesmo percurso, sem troca automática. |

**Pergunta da aula:** qual hipótese explica melhor os sinais, o que a sustenta, o que a enfraquece e o que falta verificar?

O incidente fictício ocorreu em **03/11/2026**. A aula acontece em **12/11/2026**. Os dados não representam sistemas reais do Banco do Brasil.


## 2. Preparar o projeto

### A. Localizar responsabilidades no projeto

| Arquivo | Responsabilidade | Quando olhar |
|---|---|---|
| [dados.py](projeto/src/aiops_rca/dados.py) | Carregar o caso e aplicar o corte de conhecimento. | Para entender quais registros chegaram ao modelo. |
| [rag.py](projeto/src/aiops_rca/rag.py) | Separar trechos, criar embeddings e consultar Chroma. | Para seguir um documento até o contexto. |
| [modelos.py](projeto/src/aiops_rca/modelos.py) | Montar mensagens e chamar OpenAI ou Ollama. | Para localizar a inferência real. |
| [contratos.py](projeto/src/aiops_rca/contratos.py) | Definir o formato da análise e das verificações. | Para entender o que esperamos da resposta. |
| [workflow.py](projeto/src/aiops_rca/workflow.py) | Implementar nós e ligar caminhos do LangGraph. | Para ver o grafo e acrescentar uma capacidade. |
| [sessao.py](projeto/src/aiops_rca/sessao.py) | Iniciar, consultar e retomar um checkpoint. | Para distinguir leitura de nova inferência. |
| [data/](projeto/data/) | Dossiês, registros e runbooks sintéticos. | Para conferir os insumos. |
| [dados_locais/](projeto/dados_locais/README.md) | Índice, modelos de embedding e checkpoints. | Para saber onde a execução fica guardada. |

### B. Preparar o ambiente no PowerShell

No VS Code, abra **Terminal → Novo Terminal** e selecione **PowerShell**. Execute os passos abaixo, um por vez, no mesmo terminal. Estes comandos não são células Python.

#### 1. Entrar na pasta do projeto

> **TERMINAL · POWERSHELL**  
> **Onde executar:** na pasta `03-rca-assistido`, onde estão os dois notebooks.
>
> ```powershell
> cd .\projeto
> ```
>
> **Confira:** o caminho exibido no terminal termina em `03-rca-assistido\projeto`. Os próximos comandos serão executados nessa pasta.

#### 2. Preparar o ambiente Python

> **TERMINAL · POWERSHELL**  
> **Onde executar:** na pasta `projeto`.
>
> ```powershell
> uv sync --python 3.12
> ```
>
> **Confira:** o `uv` conclui a instalação das dependências de `uv.lock` no ambiente `.venv`. Aguarde o término antes do próximo passo.

#### 3. Ativar o ambiente

> **TERMINAL · POWERSHELL**  
> **Onde executar:** na pasta `projeto`, no mesmo terminal.
>
> ```powershell
> .\.venv\Scripts\Activate.ps1
> ```
>
> **Confira:** a ativação termina sem erro; normalmente aparece `(.venv)` no início da linha. Essa seleção vale para este terminal. O kernel do notebook será escolhido no passo 5.

#### 4. Criar a configuração local, se necessário

> **TERMINAL · POWERSHELL**  
> **Onde executar:** na pasta `projeto`.
>
> ```powershell
> if (-not (Test-Path ..\.env)) { Copy-Item ..\.env.example ..\.env }
> ```
>
> **Confira:** o `.env` aparece ao lado dos notebooks, na raiz da aula. Se já estiver configurado, o comando o preserva. É normal o terminal não mostrar uma mensagem.

Abra o `.env` na raiz da aula para configurar a chave OpenAI ou o endereço do Ollama, conforme a escolha da seção C. Não cole a chave no notebook nem projete esse arquivo durante a aula.

#### 5. Selecionar o Python do notebook

Este passo é feito no **VS Code**, sem comando de terminal.

| Onde clicar | O que selecionar | Como conferir |
|---|---|---|
| **Selecionar Kernel**, no canto superior direito do notebook. | O Python do ambiente `projeto/.venv/Scripts/python.exe`. | A primeira célula de preparação usa o ambiente deste projeto. Ativar a venv no terminal, sozinho, não troca o kernel. |

### C. Escolher onde acontece a inferência

Esta aula acontece **no notebook**. Não há um aplicativo web para iniciar.

| Escolha | O que precisa estar pronto | O que permanece igual |
|---|---|---|
| **OpenAI** | Chave no `.env` e acesso à API. | Dados, índice local, grafo, contrato e revisão humana. |
| **Ollama** | Servidor disponível e `qwen2.5:7b` instalado. | Dados, índice local, grafo, contrato e revisão humana. |
| **Embeddings, nos dois caminhos** | Download explícito na seção 8; depois cache local. | Mesmo modelo de embeddings e procedimento de recuperação. |

### D. Documentação para consultar

| Documentação | Para estudar |
|---|---|
| [LangGraph](https://docs.langchain.com/oss/python/langgraph/overview) | Estado, nós, caminhos e pausas. |
| [Ollama](https://docs.ollama.com/) | Execução local e saída estruturada. |
| [OpenAI](https://developers.openai.com/api/docs/guides/structured-outputs) | Respostas estruturadas. |
| [Chroma](https://docs.trychroma.com/) | Índice e consulta vetorial. |

### Escolher onde o modelo será executado

| Antes de executar | Nesta célula |
|---|---|
| **Ação** | **Leitura local** |
| **Arquivo / função** | [ClienteModelo](projeto/src/aiops_rca/modelos.py) |
| **Entrada** | Provedor escolhido e .env privado |
| **O que observar** | Nenhuma chamada ao LLM nesta célula. |

In [ ]:
from pathlib import Path
import sys, json, inspect
import pandas as pd
from IPython.display import display, Markdown, Image

PASTA_AULA = Path.cwd() if (Path.cwd() / "projeto").is_dir() else Path.cwd().parent
PROJETO = PASTA_AULA / "projeto"
assert (PROJETO / "src" / "aiops_rca").is_dir(), "Abra o notebook na pasta da Aula 03."
sys.path.insert(0, str(PROJETO / "src"))
from aiops_rca.dados import entrada_pix, selecionar_contexto, ler_json, DATA, LOCAIS
from aiops_rca.modelos import ClienteModelo
from aiops_rca.rag import IndiceDocumental, carregar_documentos, CONSULTAS
from aiops_rca.sessao import SessaoRCA
from aiops_rca.visual import mostrar_resultado, matriz, tabela_fontes

PROVEDOR = "openai"  # Troque para "ollama" para usar o modelo local.
cliente = ClienteModelo(PROVEDOR)
pd.set_option("display.max_colwidth", 110)
print(f"Modelo escolhido: {cliente.provedor} / {cliente.modelo}")

## 3. Abrir o contexto recebido

**Pergunta:** quais informações a Aula 03 realmente recebe?

Na Aula 02, `dossie = estado["dossie"]` leu um resultado que o nó `montar_dossie` já havia gravado no estado. A função `preparar_contexto_rca` separou os registros ligados à fila.

<img src="projeto/assets/10-contexto-visual.png" width="760" style="max-width:100%;height:auto" alt="Fluxo da triagem recebida ao enriquecimento e à seleção do contexto enviado ao modelo.">

[Fonte editável](projeto/assets/10-contexto-visual.svg).

### Três conjuntos, responsabilidades diferentes

| Objeto | O que contém | O que não contém |
|---|---|---|
| Contexto compacto | Janela, corte, taxa, evidências relacionadas e lacunas. | Todos os horários e contrapontos. |
| Dossiê completo | Medição, critérios, classificações, fila e registros recebidos. | Eventos que chegaram depois do corte original. |
| Dados adicionais | Registros posteriores e trace anexado. | Confirmação automática de causa. |

### O que entra e o que fica na auditoria?

| Registro | Tratamento | Motivo |
|---|---|---|
| **E08 · parceiro externo** | Recuperado do dossiê completo como contraponto. | Não estava na lista compacta preparada no exercício. |
| **E03 · duplicata** | Conservado entre exclusões locais; fora do pacote do LLM. | Não oferecer a cópia como outra fonte da análise. |
| **E05 · cache de marketing** | Conservado entre exclusões locais; fora do pacote do LLM. | Não pertence à jornada conhecida. |

O enriquecimento é explícito. Este notebook carrega seus próprios dados; não lê automaticamente variáveis de outro notebook.

E03 e E05 continuam na lista local de exclusões, com seus motivos. A função `contexto_para_modelo()`, em [workflow.py](projeto/src/aiops_rca/workflow.py), retira essa lista do pacote enviado ao LLM. Assim podemos auditar o descarte sem oferecer a duplicata e o serviço externo à jornada como fontes da análise.

### Ler o pacote da Aula 02

| Antes de executar | Nesta célula |
|---|---|
| **Ação** | **Leitura local** |
| **Arquivo / função** | [entrada_pix](projeto/src/aiops_rca/dados.py) |
| **Entrada** | Dossiê das 10h06 e corte retrospectivo 10h25 |
| **O que observar** | A medição original permanece 8%. |

In [ ]:
entrada = entrada_pix("10:06", "10:25", incluir_trace=True)
pacote = entrada["entrada"]
display(pd.DataFrame(pacote["contexto_rca"]["evidencias"]))
display(pd.DataFrame([pacote["dossie"]["medicao"]]))
assert pacote["contexto_rca"]["taxa_erro"] == 0.08

### Enriquecer preservando a origem

| Antes de executar | Nesta célula |
|---|---|
| **Ação** | **Leitura local** |
| **Arquivo / função** | [selecionar_contexto](projeto/src/aiops_rca/dados.py) |
| **Entrada** | Pacote anterior e registros locais |
| **O que observar** | E08 aparece como contraponto; E03 e E05 ficam separados. |

In [ ]:
contexto = selecionar_contexto(pacote)
display(pd.DataFrame(contexto["evidencias"])[["id", "servico", "observacao", "classe_triagem"]])
display(pd.DataFrame(contexto["exclusoes"]))

## 4. Reconstruir a linha do tempo

**Pergunta:** estamos falando da hora do acontecimento ou da hora em que soubemos dele?

| Conceito | O que representa | Exemplo do caso |
|---|---|---|
| **Ocorrência** | Quando a observação foi feita. | CPU do banco observada às 10h01. |
| **Recebimento** | Quando o registro ficou disponível. | E09 chegou às 10h06. |
| **Janela** | Intervalo usado no cálculo. | 10h00–10h05: 80 falhas em 1.000 solicitações. |
| **Corte** | Até quando podemos conhecer os registros. | Triagem às 10h06; investigação retrospectiva até 10h25. |

<img src="projeto/assets/08-cortes.png" width="760" style="max-width:100%;height:auto" alt="Cortes de conhecimento e janela original.">

[Fonte editável](projeto/assets/08-cortes.mmd).

**O que observar:** E09 acrescenta uma evidência. Ele não altera retroativamente os **8%** calculados para a janela original.


### Conferir o que muda às 10h06

| Antes de executar | Nesta célula |
|---|---|
| **Ação** | **Leitura local** |
| **Arquivo / função** | [selecionar_contexto](projeto/src/aiops_rca/dados.py) |
| **Entrada** | Mesmo alvo, cortes diferentes |
| **O que observar** | E09 entra; taxa e janela não mudam. |

In [ ]:
c05 = selecionar_contexto(entrada_pix("10:05", "10:05", incluir_trace=False)["entrada"])
c06 = selecionar_contexto(entrada_pix("10:06", "10:06", incluir_trace=False)["entrada"])
assert c05["medicao_original"]["taxa_erro"] == c06["medicao_original"]["taxa_erro"] == 0.08
print("Novos IDs:", {e["id"] for e in c06["evidencias"]} - {e["id"] for e in c05["evidencias"]})
display(pd.DataFrame(contexto["metricas"])[["id", "inicio", "fim", "falhas", "solicitacoes", "taxa_erro", "p95_ms"]])

## 5. Ler o trace T01

**Pergunta:** onde esta requisição gastou tempo?

Um **trace** acompanha uma requisição. Cada **span** representa uma etapa. Aqui as durações são inclusivas: a etapa maior já contém as etapas internas.

<img src="projeto/assets/14-trace-visual.png" width="760" style="max-width:100%;height:auto" alt="Caixas aninhadas: requisição de 920 ms, consulta de limites de 800 ms, espera de 650 ms e consulta ao banco de 20 ms.">

[Fonte editável](projeto/assets/14-trace-visual.svg).

| Leitura | O que permite dizer | O que não permite dizer |
|---|---|---|
| **920 ms na requisição** | Duração desta requisição observada. | Que esse é o p95 de todas as requisições. |
| **650 ms esperando conexão** | A espera consumiu parte importante desta execução. | Por que a conexão demorou a ficar disponível. |
| **20 ms consultando o banco** | Duração desta consulta indicada no trace. | Que todas as consultas ou todo o banco estavam saudáveis. |

**Limite do registro:** não sabemos o resultado da requisição nem quando o trace foi recebido. Ele entra como anexo da investigação retrospectiva. As etapas exibidas não detalham todos os intervalos da execução.

[Relações do trace em Mermaid](projeto/assets/05-trace.mmd).


### Ver os spans sem somar pai e filhos

| Antes de executar | Nesta célula |
|---|---|
| **Ação** | **Leitura local** |
| **Arquivo / função** | [trace-t01.json](projeto/data/trace-t01.json) |
| **Entrada** | Trace anexado |
| **O que observar** | 650 ms dentro de 800 ms, dentro de 920 ms. |

In [ ]:
display(pd.DataFrame(contexto["trace"]["spans"]))
print("Resultado da requisição:", contexto["trace"]["resultado_requisicao"])
print("Recebimento:", contexto["trace"]["recebido_em"])

## 6. Formular perguntas de investigação

**Pergunta:** quais mecanismos diferentes poderiam produzir os sinais?

<img src="projeto/assets/11-raciocinio-visual.png" width="760" style="max-width:100%;height:auto" alt="Fluxo de observação para hipótese e para a próxima verificação.">

[Fonte editável](projeto/assets/11-raciocinio-visual.svg).

### Hipóteses concorrentes

| Hipótese | Pergunta | Evidência que precisamos confrontar |
|---|---|---|
| H1 · Pool e retenção | As conexões acabaram por capacidade menor ou por ocupação mais longa? | Mudança do pool, fila, espera e tempo de retenção. |
| H2 · Banco | Existe contenção que as amostras disponíveis não mostram? | CPU, consultas da jornada, locks e I/O. |
| H3 · Parceiro | Há falha localizada apesar de não existir relato de degradação? | Chamadas efetivas da jornada ao parceiro. |

**Fato:** há solicitações esperando. **Hipótese:** a capacidade do pool pode ter contribuído. **Conclusão:** precisa indicar a sustentação e os limites. Não vamos inserir um percentual de confiança inventado.

## 7. Conhecer a documentação

Um **runbook** registra como examinar uma condição: o que observar, quais medidas comparar e quais cuidados tomar. Ele não é um relato do que aconteceu neste incidente.

| Fonte | Pergunta que responde | Exemplo de uso |
|---|---|---|
| **Evento ou métrica** | O que observamos e quando? | Medir impacto e localizar mudanças registradas. |
| **Trace** | Onde uma requisição gastou tempo? | Localizar a espera por conexão. |
| **Runbook** | Como investigar esse mecanismo? | Identificar medidas de ocupação e retenção. |

**Cuidado na interpretação:** um runbook descrever retenção de conexão não prova que houve retenção neste incidente.

Os cinco documentos são sintéticos e estão identificados como material do laboratório. Não contêm um postmortem pronto.

### Abrir um runbook antes de indexar

| Antes de executar | Nesta célula |
|---|---|
| **Ação** | **Leitura local** |
| **Arquivo / função** | [carregar_documentos](projeto/src/aiops_rca/rag.py) |
| **Entrada** | Arquivos Markdown do corpus |
| **O que observar** | Metadados e conteúdo legível. |

In [ ]:
documentos = carregar_documentos()
display(pd.DataFrame([{k:v for k,v in d.items() if k != "texto"} for d in documentos]))
display(Markdown(documentos[1]["texto"]))

## 8. Construir e consultar o índice

**Pergunta:** como um documento vira contexto útil para a investigação?

| Peça | Função neste projeto | Não confundir com |
|---|---|---|
| **Trecho** | Unidade de texto com ID e origem. | Um documento inteiro sem rastreabilidade. |
| **Embedding** | Representação numérica usada para comparar textos. | Uma resposta gerada ou uma medida de verdade. |
| **Chroma** | Guarda e consulta vetores, textos e metadados. | O modelo que escreve as hipóteses. |
| **RAG** | Recupera documentação e a fornece à geração. | Treinar o modelo com os documentos. |

### 8.1. Preparar o índice · documentos para vetores

<img src="projeto/assets/03-rag.png" alt="Documentos, trechos, embeddings e índice" width="760" style="max-width:100%;height:auto">

[Fonte Mermaid editável](projeto/assets/03-rag.mmd).

### 8.2. Consultar · perguntas para trechos

<img src="projeto/assets/03b-busca.png" alt="Consultas, embeddings, filtro prévio e recuperação" width="760" style="max-width:100%;height:auto">

[Fonte Mermaid editável](projeto/assets/03b-busca.mmd).

### 8.3. Selecionar · trechos para a análise

<img src="projeto/assets/03c-contexto.png" alt="Intercalar rankings, deduplicar, selecionar e conferir citações" width="760" style="max-width:100%;height:auto">

[Fonte Mermaid editável](projeto/assets/03c-contexto.mmd).

O manifesto identifica corpus, modelo e regra de fragmentação. Uma alteração gera uma coleção compatível diferente; o índice anterior não é reutilizado silenciosamente.

| Limite do percurso | Valor no laboratório | O que conferir |
|---|---|---|
| Consultas | **3**: pool, banco e parceiro. | São perguntas explícitas de investigação. |
| Recuperação | **Até 3 trechos por consulta.** | Até nove ocorrências; um trecho pode aparecer em várias buscas. |
| Seleção | **Até 6 trechos distintos.** | Deduplicação por ID, origem e descartes registrados. |

**Interpretação:** distância indica proximidade na busca. A utilidade do trecho depende de ele ajudar a responder a pergunta; não é confiança na causa.


### Preparar o índice explicitamente

| Antes de executar | Nesta célula |
|---|---|
| **Ação** | **Embeddings locais; primeiro uso pode baixar o modelo** |
| **Arquivo / função** | [IndiceDocumental.preparar](projeto/src/aiops_rca/rag.py) |
| **Entrada** | Cinco documentos; embedding local multilíngue |
| **O que observar** | Primeiro uso baixa o modelo; usos seguintes reaproveitam a coleção compatível. |

In [ ]:
indice = IndiceDocumental()
# O índice pode ser reutilizado quando os documentos e a configuração coincidem.
manifesto = indice.preparar(documentos)
display(pd.DataFrame([manifesto]))

### Examinar consultas, ocorrências e deduplicação

| Antes de executar | Nesta célula |
|---|---|
| **Ação** | **Embeddings locais; sem chamada ao LLM** |
| **Arquivo / função** | [IndiceDocumental.buscar](projeto/src/aiops_rca/rag.py) |
| **Entrada** | Três perguntas explícitas |
| **O que observar** | Até nove ocorrências, até seis trechos distintos selecionados. |

In [ ]:
busca = indice.buscar()
display(pd.DataFrame(busca["ocorrencias"]))
print("Duplicatas:", busca["duplicatas"], "| Selecionados:", len(busca["selecionados"]))
display(pd.DataFrame([{"id":t["id"], "documento":t["metadados"]["titulo"], "trecho":t["texto"]} for t in busca["selecionados"]]))

## 9. Preparar a análise estruturada

**Pergunta:** como tornar a resposta organizada o suficiente para conferir?

<img src="projeto/assets/12-validacao-visual.png" width="760" style="max-width:100%;height:auto" alt="Três verificações: formato, existência das referências e sustentação da interpretação.">

[Fonte editável](projeto/assets/12-validacao-visual.svg).

### O que a resposta precisa conter

| Campo | Responsabilidade |
|---|---|
| `favoraveis` / `contrapontos` | IDs de observações que sustentam ou limitam a hipótese. |
| `documentos` | IDs dos trechos efetivamente enviados ao LLM. |
| `mecanismo` / `limites` | Explicação e alcance da interpretação. |
| `falta_verificar` | Informação que ainda precisamos obter. |
| `acoes_propostas` | Investigação sugerida, com papel responsável e evidência de conclusão. |

Pydantic valida a estrutura. A checagem de referências valida a existência das fontes. A pessoa ainda precisa avaliar se a fonte sustenta a frase.

### Ver o contrato, o prompt e a chamada

| Antes de executar | Nesta célula |
|---|---|
| **Ação** | **Leitura local** |
| **Arquivo / função** | [AnaliseRCA](projeto/src/aiops_rca/contratos.py) e [ClienteModelo.gerar](projeto/src/aiops_rca/modelos.py) |
| **Entrada** | Código do projeto |
| **O que observar** | O ponto que envia mensagens aparece em gerar; esta célula apenas lê. |

In [ ]:
from aiops_rca.contratos import AnaliseRCA
from aiops_rca.modelos import INSTRUCAO
print(INSTRUCAO)
display(pd.DataFrame([{"campo": nome, "descrição": campo.description or str(campo.annotation)}
                     for nome, campo in AnaliseRCA.model_fields.items()]))
display(Markdown("```python\n" + inspect.getsource(ClienteModelo.gerar) + "\n```"))

## 10. Montar e executar o grafo

**Pergunta:** qual função executa cada responsabilidade e onde o caminho muda?

### Visão dos componentes

O desenho mostra onde estão dados, índice, modelo e checkpoint. Em seguida abrimos o grafo com todos os nós e caminhos.

<img src="projeto/assets/02-arquitetura.png" alt="Notebook, serviço, dados, Chroma, modelo e checkpoint" width="760" style="max-width:100%;height:auto">

[Fonte Mermaid editável](projeto/assets/02-arquitetura.mmd).

### Grafo completo · funções e caminhos reais

<img src="projeto/assets/04-grafo.png" alt="Todos os nós e rotas do grafo RCA" width="760" style="max-width:100%;height:auto">

[Fonte Mermaid editável](projeto/assets/04-grafo.mmd).

### Vocabulário para ler a montagem

| Operação LangGraph | O que faz aqui |
|---|---|
| `StateGraph(EstadoRCA)` | Define quais dados circulam entre nós. |
| `add_node(nome, funcao)` | Associa uma responsabilidade a uma função Python. |
| `add_edge(origem, destino)` | Define uma passagem incondicional. |
| `add_conditional_edges(...)` | Escolhe o caminho pelo estado calculado. |
| `compile(checkpointer=...)` | Prepara execução e persistência. |
| `invoke(entrada, config)` | Executa até terminar ou pausar. |

### Ler a montagem e visualizar o grafo

| Antes de executar | Nesta célula |
|---|---|
| **Ação** | **Leitura local** |
| **Arquivo / função** | [configurar_fluxo_rca](projeto/src/aiops_rca/workflow.py) |
| **Entrada** | Cliente e índice já preparados |
| **O que observar** | Todos os nós, condições e o ponto de revisão. |

In [ ]:
from aiops_rca.workflow import configurar_fluxo_rca, conectar_validacao, montar_grafo
sessao = SessaoRCA(cliente, indice)
display(Markdown("```python\n" + inspect.getsource(configurar_fluxo_rca) + "\n```"))
# O desenho Mermaid vem da estrutura real do grafo. Não chama modelos.
print(sessao.grafo.get_graph().draw_mermaid())
display(Image(filename=str(PROJETO / "assets" / "04-grafo.png"), width=760))

### Executar a investigação até a pausa

| Antes de executar | Nesta célula |
|---|---|
| **Ação** | **Chamada ao LLM; salva checkpoint** |
| **Arquivo / função** | [SessaoRCA.iniciar](projeto/src/aiops_rca/sessao.py) |
| **Entrada** | Entrada retrospectiva, índice e provedor |
| **O que observar** | Uma análise real; resultado deve aguardar revisão. |

In [ ]:
# Esta chamada inicia a análise e guarda a pausa para revisão.
ID_EXECUCAO, estado = sessao.iniciar(entrada)
print("Execução:", ID_EXECUCAO)
mostrar_resultado(estado)
assert estado["status"] == "aguardando_revisao", "Leia erro/validação antes de continuar. Não há aprovação silenciosa."

## 11. Conferir e revisar o postmortem

**Pergunta:** o que preciso ler antes de aprovar a versão apresentada?

| Conferência | Onde olhar | Pergunta de revisão |
|---|---|---|
| **Contexto enviado** | Trechos selecionados e observações. | O modelo recebeu informação suficiente? |
| **Fonte citada** | IDs da matriz e tabela de fontes. | A fonte citada estava no contexto? |
| **Interpretação** | Afirmação da hipótese e texto original. | A observação sustenta essa afirmação? |

<img src="projeto/assets/13-revisao-visual.png" width="760" style="max-width:100%;height:auto" alt="Produzir análise, persistir e pausar, depois aprovar ou pedir revisão.">

[Fonte editável](projeto/assets/13-revisao-visual.svg).

### Exemplo fictício para criticar

Exemplo de resposta fictícia incorreta para discussão: “A causa está confirmada: CPU do banco a 95%. Reinicie o banco e apague os logs.”

O contexto não sustenta esses 95%. A recomendação destrói evidências e pula a investigação. Não é uma saída real apresentada como resultado do sistema; é um contraexemplo para a turma criticar.

### Uma checagem na resposta real

 procure E09 na hipótese do banco. Esse registro informa CPU de 35%, observada às 10h01 e recebida às 10h06. O atraso de recebimento não é lentidão de processamento. Se o modelo usar E09 como evidência favorável a contenção, peça que reveja essa classificação. A amostra também não basta para declarar todo o banco saudável.
### Sequência completa · chamada, checkpoint e retomada

<img src="projeto/assets/06-revisao.png" alt="Pausa, checkpoint, consulta e decisão humana" width="760" style="max-width:100%;height:auto">

[Fonte Mermaid editável](projeto/assets/06-revisao.mmd).

### Conferir fontes e ações propostas

| Antes de executar | Nesta célula |
|---|---|
| **Ação** | **Leitura local** |
| **Arquivo / função** | [tabela_fontes](projeto/src/aiops_rca/visual.py) |
| **Entrada** | Resultado persistido |
| **O que observar** | IDs citados na matriz devem existir nesta tabela ou nas observações. |

In [ ]:
display(matriz(estado))
display(tabela_fontes(estado))
display(pd.DataFrame(estado["linha_tempo"]))
display(pd.DataFrame([estado["postmortem"]["impacto"]]))
display(pd.DataFrame(estado["postmortem"]["mitigacao_observada"]))
display(pd.DataFrame({"Limitações": estado["postmortem"]["limitacoes"]}))
display(pd.DataFrame(estado["postmortem"]["acoes_propostas"]))
print("Versão apresentada:", estado["versao"])

### Ler o checkpoint sem fazer outra análise

| Antes de executar | Nesta célula |
|---|---|
| **Ação** | **Leitura local** |
| **Arquivo / função** | [SessaoRCA.consultar](projeto/src/aiops_rca/sessao.py) |
| **Entrada** | Mesmo identificador de execução |
| **O que observar** | Quantidade de chamadas permanece igual. |

In [ ]:
antes = len(estado["chamadas"])
# O ID permite ler o estado da execução salva.
salvo = sessao.consultar(ID_EXECUCAO)
assert len(salvo.values["chamadas"]) == antes
print("Próximo nó:", salvo.next, "| Chamadas realizadas:", antes)

### Registrar uma decisão humana

| Antes de executar | Nesta célula |
|---|---|
| **Ação** | **Decisão humana; nova inferência somente ao solicitar revisão** |
| **Arquivo / função** | [SessaoRCA.decidir](projeto/src/aiops_rca/sessao.py) |
| **Entrada** | ID, versão lida, ação e comentário |
| **O que observar** | Aprovação finaliza; revisão faz outra análise e pausa novamente. |

In [ ]:
DECISAO = None  # Depois de ler: "aprovar" ou "solicitar_revisao".
COMENTARIO = ""  # Obrigatório se solicitar revisão.
if DECISAO is not None:
    versao_lida = estado["versao"]  # Versão do rascunho que acabamos de ler.
    estado = sessao.decidir(ID_EXECUCAO, versao_lida, DECISAO, COMENTARIO)
    mostrar_resultado(estado)
else:
    print("Nenhuma decisão enviada. O postmortem continua aguardando sua revisão.")

## 12. Exercício médio · acrescentar um nó de investigação

**Problema:** o postmortem mostra hipóteses, mas precisamos tornar a próxima coleta mais concreta.

**Você vai construir:** um nó `propor_verificacoes`, entre a validação das referências e a montagem do postmortem. Para H1, H2 e H3, ele deve devolver pergunta, dado a coletar, resultado favorável e resultado contrário.

| Entrada | Saída | Limite |
|---|---|---|
| Análise validada, contexto e documentos | `plano_verificacoes` e registro da nova chamada | Apenas propor; nenhuma coleta ou mudança operacional é executada. |

**Pistas:** `gerar_verificacoes(estado, cliente)` já encapsula a API e valida o contrato. Seu nó precisa acrescentar a resposta ao estado, preservando o que já existe. Depois, conecte o caminho válido ao novo nó e o novo nó ao postmortem.
<img src="projeto/assets/07-exercicio.png" alt="Novo nó entre validação e postmortem" width="760" style="max-width:100%;height:auto">

[Fonte Mermaid editável](projeto/assets/07-exercicio.mmd).

### Sua implementação · estado e função

| Ação | Onde está | O que recebe | O que observar |
|---|---|---|---|
| **Leitura local** | [gerar_verificacoes](projeto/src/aiops_rca/workflow.py) | Estado com análise validada | Acrescente uma chave sem apagar o dossiê e a análise. |

In [ ]:
from aiops_rca.contratos import EstadoRCA
from aiops_rca.workflow import gerar_verificacoes

class EstadoComPlano(EstadoRCA, total=False):
    plano_verificacoes: dict | None

def propor_verificacoes(estado, cliente):
    # Receba plano e metadados de gerar_verificacoes.
    # Devolva apenas as atualizações do estado.
    raise NotImplementedError("Implemente o nó antes de executar o grafo ampliado.")

### Sua implementação · conectar o nó

| Ação | Onde está | O que recebe | O que observar |
|---|---|---|---|
| **Leitura local** | [configurar_fluxo_rca](projeto/src/aiops_rca/workflow.py) | Funções existentes e novo estado | Conecte validação, verificações e postmortem. |

In [ ]:
def montar_grafo_com_plano(cliente, indice, checkpointer):
    g = configurar_fluxo_rca(cliente, indice, tipo_estado=EstadoComPlano)
    # Adicione propor_verificacoes e conecte o caminho válido até ele.
    # Depois conecte o novo nó a montar_postmortem e compile.
    raise NotImplementedError("Complete as conexões.")

### Executar e conferir a capacidade acrescentada

| Ação | Onde está | O que recebe | O que observar |
|---|---|---|---|
| **Duas chamadas ao LLM; nova execução** | [SessaoRCA](projeto/src/aiops_rca/sessao.py) | Novo grafo, mesmo caso | Duas chamadas: análise e plano; três hipóteses cobertas. |

Complete as duas células de implementação e mude `RODAR_EXERCICIO_MEDIO` para `True` para conferir o novo nó.

In [ ]:
RODAR_EXERCICIO_MEDIO = False

if RODAR_EXERCICIO_MEDIO:
    sessao_plano = SessaoRCA(cliente, indice, construtor=montar_grafo_com_plano)
    arestas = {(a.source, a.target) for a in sessao_plano.grafo.get_graph().edges}
    assert ("validar_referencias", "propor_verificacoes") in arestas
    assert ("propor_verificacoes", "montar_postmortem") in arestas
    ID_PLANO, estado_plano = sessao_plano.iniciar(entrada)
    assert estado_plano["status"] == "aguardando_revisao", estado_plano.get("validacao")
    assert estado_plano["contexto"] == estado["contexto"]
    assert len(estado_plano["chamadas"]) == 2
    display(pd.DataFrame(estado_plano["plano_verificacoes"]["verificacoes"]))
else:
    print('Complete o nó e as conexões. Depois ative RODAR_EXERCICIO_MEDIO.')

## 13. Exercício hard · ampliar o RAG

**Problema:** a documentação adicional contém uma orientação nova, uma versão antiga e um serviço diferente. Uma busca por palavras parecidas pode trazer todos.

**Você vai construir:** uma função de recuperação que filtre a versão vigente e a jornada Pix **antes** de selecionar os três resultados mais próximos.

| Arquivo adicional | Metadados para conferir |
|---|---|
| Runbook de retenção v2 | `vigente=True`, `jornada="pix"` |
| Runbook anterior v1 | `vigente=False` |
| Pool de marketing | `jornada="marketing"` |

**Pistas:** a biblioteca Chroma aceita `where` com `$and` e `$eq`. Nosso método `buscar` recebe esse filtro e o aplica antes do top-k. Faça uma pergunta específica sobre empréstimo e devolução da conexão; depois confira o documento recuperado, sem obrigar a IA a uma conclusão diferente.

Complete a função de recuperação e mude `RODAR_EXERCICIO_HARD` para `True` na célula de indexação. Para rodar o grafo ampliado, conclua também o exercício médio.

### Conhecer e indexar o pacote adicional

| Ação | Onde está | O que recebe | O que observar |
|---|---|---|---|
| **Embeddings locais; sem chamada ao LLM** | [carregar_documentos](projeto/src/aiops_rca/rag.py) | Corpus original e três documentos adicionais | Nova coleção compatível; índice original preservado. |

In [ ]:
RODAR_EXERCICIO_HARD = False

if RODAR_EXERCICIO_HARD:
    extras = carregar_documentos(DATA / "documentos_exercicio")
    display(pd.DataFrame([{k:v for k,v in d.items() if k != "texto"} for d in extras]))
    indice_ampliado = IndiceDocumental()
    display(pd.DataFrame([indice_ampliado.preparar(documentos + extras)]))
else:
    print('Ative RODAR_EXERCICIO_HARD para indexar o pacote adicional.')

### Sua implementação · recuperar documentação aplicável

| Ação | Onde está | O que recebe | O que observar |
|---|---|---|---|
| **Leitura local** | [IndiceDocumental.buscar](projeto/src/aiops_rca/rag.py) | Índice ampliado | Filtro de vigência e jornada aplicado antes dos resultados. |

In [ ]:
def recuperar_documentacao_aplicavel(indice):
    # Monte filtro com $and: vigente == True e jornada == "pix".
    # Consulte retenção, banco e parceiro; devolva o resultado de buscar.
    raise NotImplementedError("Implemente a recuperação filtrada.")

### Testar a recuperação antes de chamar o modelo

| Ação | Onde está | O que recebe | O que observar |
|---|---|---|---|
| **Embeddings locais** | [IndiceDocumental.buscar](projeto/src/aiops_rca/rag.py) | Índice ampliado | Runbook v2 presente; anterior e marketing ausentes. |

In [ ]:
if RODAR_EXERCICIO_HARD:
    busca_ampliada = recuperar_documentacao_aplicavel(indice_ampliado)
    selecionados = busca_ampliada["selecionados"]
    assert all(t["metadados"]["vigente"] and t["metadados"]["jornada"] == "pix" for t in busca_ampliada["candidatos"])
    assert any(t["id"].startswith("D06-v2-") for t in selecionados), "Confira consulta e trecho antes de analisar."
    display(pd.DataFrame([{"id":t["id"], "documento":t["metadados"]["titulo"], "trecho":t["texto"]} for t in selecionados]))

### Aplicar a recuperação ampliada ao grafo

| Ação | Onde está | O que recebe | O que observar |
|---|---|---|---|
| **Duas chamadas ao LLM; nova execução** | [configurar_fluxo_rca](projeto/src/aiops_rca/workflow.py) | Novo retriever e nó do exercício médio | Mesmo incidente; nova documentação; revisão humana mantida. |

In [ ]:
if RODAR_EXERCICIO_HARD:
    if "estado_plano" not in globals():
        raise RuntimeError("Conclua o exercício médio antes de rodar o grafo ampliado.")

    def montar_grafo_ampliado(cliente, indice, checkpointer):
        g = configurar_fluxo_rca(cliente, indice, tipo_estado=EstadoComPlano,
                                recuperar=recuperar_documentacao_aplicavel)
        from functools import partial
        g.add_node("propor_verificacoes", partial(propor_verificacoes, cliente=cliente))
        conectar_validacao(g, "propor_verificacoes")
        g.add_edge("propor_verificacoes", "montar_postmortem")
        return g.compile(checkpointer=checkpointer)

    sessao_ampliada = SessaoRCA(cliente, indice_ampliado, construtor=montar_grafo_ampliado)
    ID_AMPLIADA, estado_ampliado = sessao_ampliada.iniciar(entrada)
    assert estado_ampliado["status"] == "aguardando_revisao", estado_ampliado.get("validacao")
    assert estado_ampliado["contexto"] == estado_plano["contexto"]
    mostrar_resultado(estado_ampliado)
    display(pd.DataFrame(estado_ampliado["plano_verificacoes"]["verificacoes"]))

## 14. Fechamento

| Pergunta de conferência | Onde responder |
|---|---|
| De onde vieram as 80 falhas? | Série de métricas e medição original da Aula 02. |
| Por que E03 não aumentou o impacto? | É uma duplicata conhecida de E02. |
| O que os 650 ms permitem dizer? | Houve espera por conexão nessa requisição; falta explicar seu mecanismo. |
| O que o RAG acrescentou? | Orientações documentais para interpretar e investigar. |
| O que a validação técnica não resolveu? | Se cada interpretação é sustentada e suficiente. |
| Quem aprovou o postmortem? | Somente a pessoa que enviou a decisão da versão apresentada. |

**Para a Aula 04:** já sabemos produzir uma análise revisável. O próximo tema será disponibilizar consultas e propostas com controles de acesso e limites de ação.

**Limites deste laboratório:** cenário sintético, corpus pequeno, hipóteses delimitadas e um trace. Não há monitoramento contínuo, execução de remediação ou confirmação automática de causa. LangSmith não é necessário nesta etapa; checkpoints e registros de chamadas são locais.